In [1]:
from datetime import date

import nflreadpy as nfl
import pandas as pd

current_year = date.today().year
seasons = list(range(current_year - 4, current_year + 1))
print("Requested seasons:", seasons)

Requested seasons: [2022, 2023, 2024, 2025, 2026]


In [2]:
from nfl_analyzer.filters import (
    aggregate_by_player_id,
    exclude_incoming_rookies,
    exclude_significant_injury,
    filter_draftable_positions,
    skill_depth_ids,
    starting_qb_ids,
)


def load_stats(load_fn, seasons: list[int]) -> pd.DataFrame:
    """Load stats for as many of the requested seasons as are available."""
    frames: list[pd.DataFrame] = []
    loaded: list[int] = []
    for season in seasons:
        try:
            frame = load_fn(season).to_pandas()
        except Exception as exc:  # noqa: BLE001 - missing seasons are expected
            print(f"Skipping {season}: {exc}")
            continue
        if frame.empty:
            print(f"Skipping {season}: no rows returned")
            continue
        frames.append(frame)
        loaded.append(season)
    if not frames:
        raise RuntimeError("No seasons returned data")
    print("Loaded seasons:", loaded)
    return pd.concat(frames, ignore_index=True)


team_stats = load_stats(nfl.load_team_stats, seasons)
player_stats = load_stats(nfl.load_player_stats, seasons)
print("team_stats shape:", team_stats.shape)
print("player_stats shape (before inactive filter):", player_stats.shape)

players = nfl.load_players().to_pandas()
latest_loaded_season = int(player_stats["season"].max())
eligible_players = players[
    (players["status"] != "RET")
    & (players["last_season"].fillna(0) >= latest_loaded_season)
][["gsis_id"]].dropna()

player_stats = player_stats.merge(
    eligible_players,
    left_on="player_id",
    right_on="gsis_id",
    how="inner",
).drop(columns=["gsis_id"])
print(
    "player_stats shape (after excluding RET / stale last_season):",
    player_stats.shape,
)
print("Eligibility cutoff last_season >=", latest_loaded_season)

before = len(player_stats)
player_stats = exclude_incoming_rookies(player_stats, players, latest_loaded_season)
print(
    f"After excluding incoming rookies (draft_year > {latest_loaded_season}):",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

injuries = nfl.load_injuries(latest_loaded_season).to_pandas()
before = len(player_stats)
player_stats = exclude_significant_injury(
    player_stats, injuries, latest_loaded_season
)
print(
    f"After excluding significant injury (>2 REG Out weeks in {latest_loaded_season}):",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

before = len(player_stats)
player_stats = filter_draftable_positions(player_stats)
print(
    "After keeping draftable positions only:",
    player_stats.shape,
    f"(removed {before - len(player_stats)} rows)",
)

before = len(player_stats)
player_stats = aggregate_by_player_id(player_stats)
print(
    "After grouping by player_id:",
    player_stats.shape,
    f"(from {before} weekly rows)",
)
print("Remaining positions:\n", player_stats["position"].value_counts())

depth_charts = nfl.load_depth_charts(latest_loaded_season).to_pandas()
starter_qbs = starting_qb_ids(depth_charts)
skill_depth = skill_depth_ids(depth_charts)
print(f"Depth-chart QB starters: {len(starter_qbs)}")
print(f"Depth-chart skill players (rank <= 2): {len(skill_depth)}")


Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
team_stats shape: (2278, 138)
player_stats shape (before inactive filter): (75879, 150)
player_stats shape (after excluding RET / stale last_season): (66432, 150)
Eligibility cutoff last_season >= 2025
After excluding incoming rookies (draft_year > 2025): (66432, 150) (removed 0 rows)
After excluding significant injury (>2 REG Out weeks in 2025): (60935, 150) (rem

In [3]:
team_stats

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2022,1,ARI,REG,2022_01_KC_ARI,KC,24,38,205,2,...,255,0,0,0,0,0,5,44,0,211
1,2022,1,ATL,REG,2022_01_NO_ATL,NO,20,33,215,0,...,180,2,0,0,0,3,1,12,0,168
2,2022,1,BAL,REG,2022_01_BAL_NYJ,NYJ,17,30,213,3,...,291,3,0,0,1,3,2,30,0,241
3,2022,1,BUF,REG,2022_01_BUF_LA,LA,26,31,297,3,...,0,0,0,0,0,0,0,0,0,0
4,2022,1,CAR,REG,2022_01_CLE_CAR,CLE,16,27,235,1,...,250,0,0,0,0,0,5,29,0,221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,2025,21,LA,POST,2025_21_LA_SEA,SEA,22,35,374,3,...,84,1,0,0,0,2,0,0,0,84
2274,2025,21,NE,POST,2025_21_NE_DEN,DEN,10,21,86,0,...,303,1,3,1,0,1,3,8,0,295
2275,2025,21,SEA,POST,2025_21_LA_SEA,LA,25,36,346,3,...,226,5,0,0,0,3,2,0,0,226
2276,2025,22,NE,POST,2025_22_SEA_NE,SEA,27,43,295,2,...,356,4,0,0,0,6,2,16,0,340


In [4]:
player_stats

,player_id,player_name,player_display_name,position,position_group,headshot_url,team,games,season_min,season_max,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0022942,P.Rivers,Philip Rivers,QB,QB,https://static.www.nfl.com/image/private/f_aut...,IND,3,2025,2025,...,0,0,0,0,0,0,0,0,31.66,31.66
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,PIT,52,2022,2025,...,0,0,0,0,0,0,0,0,723.70,724.70
2,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,BUF,51,2022,2025,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0024243,M.Lewis,Marcedes Lewis,TE,TE,https://static.www.nfl.com/image/private/f_aut...,DEN,18,2022,2025,...,0,0,0,0,0,0,0,0,29.70,40.70
4,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,NYJ,64,2022,2025,...,0,0,0,1,1,3,0,78,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,00-0040737,T.Ferguson,Terrance Ferguson,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,LA,13,2025,2025,...,0,0,0,0,0,0,0,0,43.00,55.00
715,00-0040739,E.Arroyo,Elijah Arroyo,TE,TE,https://static.www.nfl.com/image/upload/f_auto...,SEA,13,2025,2025,...,0,0,0,0,0,0,0,0,21.90,36.90
716,00-0040743,T.Shough,Tyler Shough,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NO,11,2025,2025,...,0,0,0,0,0,0,0,0,157.96,157.96
717,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,CLE,15,2025,2025,...,0,0,0,0,0,0,0,0,36.70,54.70


In [5]:
qb_rankings = player_stats.loc[player_stats["position"] == "QB"].copy()
before = len(qb_rankings)
qb_rankings = qb_rankings[
    qb_rankings["player_id"].astype(str).isin(starter_qbs)
].copy()
print(f"QB starters after depth-chart filter: {len(qb_rankings)} (from {before})")
qb_rankings["total_tds"] = (
    qb_rankings["passing_tds"].fillna(0) + qb_rankings["rushing_tds"].fillna(0)
)
qb_rankings = (
    qb_rankings.sort_values("passing_yards", ascending=False)[
        [
            "player_display_name",
            "team",
            "position",
            "games",
            "passing_yards",
            "passing_tds",
            "rushing_tds",
            "total_tds",
        ]
    ]
    .reset_index(drop=True)
)
qb_rankings


QB starters after depth-chart filter: 27 (from 92)


,player_display_name,team,position,games,passing_yards,passing_tds,rushing_tds,total_tds
0,Patrick Mahomes,KC,QB,73,19381,134,13,147
1,Jared Goff,DET,QB,72,19356,135,2,137
2,Josh Allen,BUF,QB,74,18185,132,56,188
3,Matthew Stafford,LA,QB,63,16357,112,1,113
4,Justin Herbert,LAC,QB,66,16144,96,7,103
5,Geno Smith,LV,QB,65,15504,92,4,96
6,Jalen Hurts,PHI,QB,72,15409,98,60,158
7,Baker Mayfield,TB,QB,66,15271,113,6,119
8,Trevor Lawrence,JAX,QB,63,14893,94,21,115
9,Dak Prescott,DAL,QB,57,14820,108,7,115


In [6]:
kicker_rankings = (
    player_stats.loc[player_stats["position"] == "K"]
    .sort_values("fg_made", ascending=False)[
        [
            "player_display_name",
            "team",
            "position",
            "games",
            "fg_made",
            "fg_att",
        ]
    ]
    .reset_index(drop=True)
)
kicker_rankings


,player_display_name,team,position,games,fg_made,fg_att
0,Ka'imi Fairbairn,HOU,K,67,147,163
1,Jason Myers,SEA,K,72,145,166
2,Cameron Dicker,LAC,K,65,135,145
3,Chase McLaughlin,TB,K,70,127,144
4,Harrison Butker,KC,K,70,126,144
5,Chris Boswell,PIT,K,66,120,138
6,Wil Lutz,DEN,K,71,116,138
7,Daniel Carlson,LV,K,68,116,134
8,Jake Elliott,PHI,K,76,115,136
9,Brandon Aubrey,DAL,K,52,113,128


In [7]:
skill_rankings = player_stats.loc[
    player_stats["position"].isin(["RB", "WR", "TE", "FB"])
].copy()
before = len(skill_rankings)
skill_rankings = skill_rankings[
    skill_rankings["player_id"].astype(str).isin(skill_depth)
].copy()
print(
    f"Skill players after depth-chart filter (rank <= 2): "
    f"{len(skill_rankings)} (from {before})"
)
skill_rankings["total_yards"] = (
    skill_rankings["rushing_yards"].fillna(0)
    + skill_rankings["receiving_yards"].fillna(0)
)
skill_rankings = (
    skill_rankings.sort_values("total_yards", ascending=False)[
        [
            "player_display_name",
            "team",
            "position",
            "games",
            "rushing_yards",
            "receiving_yards",
            "total_yards",
        ]
    ]
    .reset_index(drop=True)
)
skill_rankings


Skill players after depth-chart filter (rank <= 2): 185 (from 582)


,player_display_name,team,position,games,rushing_yards,receiving_yards,total_yards
0,Saquon Barkley,PHI,RB,69,6138,1346,7484
1,Derrick Henry,BAL,RB,69,6491,955,7446
2,Christian McCaffrey,SF,RB,62,4591,2693,7284
3,Josh Jacobs,GB,RB,64,4852,1363,6215
4,James Cook,BUF,RB,75,4886,1293,6179
...,...,...,...,...,...,...,...
180,Jordan James,SF,RB,1,28,7,35
181,Dylan Laube,LV,RB,23,9,18,27
182,Jack Westover,NE,TE,5,0,8,8
183,Robbie Ouzts,SEA,RB,5,0,0,0
